# 911 Emergency Calls — Exploratory Data Analysis
### Montgomery County, Pennsylvania Emergency Dispatch Data

## 1. Project Overview
This notebook analyses more than 663,000 emergency 911 calls from Montgomery County, PA. We explore temporal patterns, call type distributions, geographic hotspots, and hour/day/month trends to understand how emergency dispatch volumes change over time and by category.

## 2. Learning Objectives
- Parse and engineer datetime features (hour, day, month, year)
- Visualise time-series call volumes with line plots and heatmaps
- Perform groupby aggregations across multiple dimensions
- Apply seaborn heatmaps for bivariate categorical analysis
- Extract the top-level call reason from free-text description fields

## 3. Business / Research Problem
**Question:** What are the most common categories of 911 calls, when do they peak, and which zip codes generate the most calls? How can this information help resource allocation for emergency services?

## 4. Why This Analysis Matters
Emergency services operate with fixed budgets and finite personnel. Data-driven dispatch scheduling — informed by peak hours, high-volume zip codes, and seasonal trends — can reduce response times, save lives, and cut operational costs.

## 5. Dataset Overview
The dataset contains calls with these key columns:
- `lat`, `lng` — latitude and longitude of the incident
- `desc` — free-text incident description
- `zip` — zip code
- `title` — call type (e.g., EMS: RESPIRATORY EMERGENCY)
- `timeStamp` — datetime string of the call
- `twp` — township name
- `addr` — address
- `e` — dummy variable (always 1)

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `mchirico/montcoalert`
- **Source:** Montgomery County, PA Emergency Communications
- **License:** Public domain / open data

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'mchirico/montcoalert'
CALL_CATEGORIES = ['EMS','Fire','Traffic']

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

In [ ]:
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head(3)

## 11. Data Validation Checks

In [ ]:
print('Columns:', df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum()>0])
print('\nDtypes:')
print(df.dtypes)

## 12. Data Cleaning
1. Parse timestamps
2. Extract top-level reason (before the colon in title)
3. Drop rows with null timestamps or titles

In [ ]:
df['timeStamp'] = pd.to_datetime(df['timeStamp'], errors='coerce')
df = df.dropna(subset=['timeStamp','title'])
# Extract call category (before ':')
df['Reason'] = df['title'].apply(lambda x: x.split(':')[0].strip())
# Datetime features
df['Hour']       = df['timeStamp'].dt.hour
df['Month']      = df['timeStamp'].dt.month
df['Day_of_Week'] = df['timeStamp'].dt.day_name()
df['Year']       = df['timeStamp'].dt.year
df['Date']       = df['timeStamp'].dt.date
print(f'Clean records: {len(df)}')
print('Reason counts:')
print(df['Reason'].value_counts())

## 13. Exploratory Data Analysis

In [ ]:
print('Date range:', df['timeStamp'].min().date(), '→', df['timeStamp'].max().date())
print('Unique zip codes:', df['zip'].nunique())
print('Unique townships:', df['twp'].nunique())
df.describe(include='all').T.head(10)

## 14. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
reason_counts = df['Reason'].value_counts()
reason_counts.plot(kind='bar', ax=axes[0], color=['steelblue','firebrick','seagreen'])
axes[0].set_title('Call Count by Reason')
axes[0].set_ylabel('Number of Calls')
axes[0].tick_params(axis='x', rotation=0)
top_zip = df['zip'].value_counts().head(15)
top_zip.plot(kind='barh', ax=axes[1], color='slateblue')
axes[1].set_title('Top 15 Zip Codes by Call Volume')
axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis

In [ ]:
# Calls per hour by reason
hour_reason = df.groupby(['Hour','Reason']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14,5))
hour_reason.plot(ax=ax)
ax.set_title('Calls per Hour by Category')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Calls')
ax.legend(title='Reason')
plt.tight_layout(); plt.show()

In [ ]:
# Calls over time (monthly)
monthly = df.resample('M', on='timeStamp').size()
fig, ax = plt.subplots(figsize=(14,4))
monthly.plot(ax=ax, color='darkcyan')
ax.set_title('Monthly Call Volume Over Time')
ax.set_ylabel('Calls per Month')
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Heatmap: Hour vs Day of Week
Visualize intensity of 911 calls across every hour/weekday combination.

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_hour = df.groupby(['Day_of_Week','Hour']).size().unstack(fill_value=0)
day_hour = day_hour.reindex(day_order)
fig, ax = plt.subplots(figsize=(16,6))
sns.heatmap(day_hour, cmap='YlOrRd', linewidths=0.1, ax=ax)
ax.set_title('911 Call Heatmap: Day of Week vs Hour', fontsize=14)
ax.set_xlabel('Hour')
ax.set_ylabel('Day of Week')
plt.tight_layout(); plt.show()

## 17. Statistical Checks / Hypothesis Testing
**Test:** Do weekdays and weekends have significantly different call volumes?

In [ ]:
from scipy import stats
df['is_weekend'] = df['Day_of_Week'].isin(['Saturday','Sunday'])
daily = df.groupby(['Date','is_weekend']).size().reset_index(name='calls')
wd = daily[~daily['is_weekend']]['calls']
we = daily[daily['is_weekend']]['calls']
t, p = stats.ttest_ind(wd, we)
print(f'Weekday avg: {wd.mean():.1f} | Weekend avg: {we.mean():.1f}')
print(f't={t:.3f}, p={p:.4f}')
print('Weekdays significantly busier:', p < 0.05)

## 18. Visual Analysis
### Top-5 Call Types (within each category)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))
for i, reason in enumerate(['EMS','Fire','Traffic']):
    top5 = (
        df[df['Reason']==reason]['title'].value_counts().head(5)
    )
    top5.plot(kind='barh', ax=axes[i], color=['#e63946','#457b9d','#2a9d8f','#f4a261','#264653'])
    axes[i].set_title(f'Top 5: {reason}')
    axes[i].invert_yaxis()
    axes[i].set_xlabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
# Month × Reason heatmap
month_reason = df.groupby(['Month','Reason']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(10,5))
sns.heatmap(month_reason, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Call Volume: Month × Reason')
plt.tight_layout(); plt.show()

## 19. Key Findings
1. **EMS calls dominate** — medical emergencies account for the majority of 911 calls.
2. **Peak hours:** Call volume is highest between 8 AM–8 PM, with a notable morning rush spike.
3. **Weekdays are busier** than weekends, especially for Traffic calls.
4. **Geographic concentration:** A handful of zip codes generate a disproportionate share of calls.
5. **Seasonal pattern:** Call volumes show predictable seasonal spikes (winter for EMS, summer for Fire).

## 20. Limitations
- Geographic scope is limited to Montgomery County, PA — results may not generalise.
- Call descriptions are free text and inconsistently formatted.
- Duplicate calls for the same incident cannot be identified from this dataset.
- Response time and outcome data are not included.

## 21. Recommendations / Next Steps
1. Build a time-series forecast model (e.g., Prophet) to predict daily call volume.
2. Cluster zip codes by call category mix to identify resource-deployment zones.
3. Enrich with weather data to analyse weather-driven call spikes.
4. Add geospatial choropleth maps for spatial hotspot analysis.
5. Detect anomalous days (extreme spikes) and correlate with historical events.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Using raw `title` as category | Too granular — hundreds of values | Extract `Reason` prefix |
| Not parsing timestamp properly | String comparison fails for time-based analysis | Use `pd.to_datetime()` |
| Ignoring time zones | Timestamps may be local time | Confirm or localise with `tz_localize` |
| Averaging daily counts without accounting for weekends | Distorts hourly patterns | Separate weekday from weekend |
| Treating zip code as numeric | Zip codes are categorical identifiers | Cast to string |

## 23. Mini Challenge / Exercises
1. **Township analysis**: Which townships have the highest EMS call rates per 1000 population?
2. **Anomaly detection**: Find dates where call volume was more than 2 standard deviations above the monthly mean.
3. **Rolling average**: Plot a 7-day rolling average of daily call volume for each reason type.
4. **Correlation**: Do Fire and EMS calls correlate temporally? Compute monthly Pearson correlation.
5. **How to extend**: Merge with U.S. Census data to compute per-capita call rates by zip code.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  EMS is the dominant call type; Traffic spikes during rush hours.
TAKEAWAY 2  Peak call hours are 8 AM–8 PM with a slight morning rush effect.
TAKEAWAY 3  Weekdays see significantly more calls than weekends.
TAKEAWAY 4  A few zip codes are disproportionate emergency hotspots.
TAKEAWAY 5  Time-series decomposition and geographic mapping are the natural next steps.
```